# Week 2 – Urdu OCR Image Preprocessing

In this notebook, I preprocess my Urdu OCR dataset by converting images to grayscale, resizing them, removing noise, and converting them into binary images. After preprocessing, I evaluate Tesseract OCR on the processed images and analyze its performance on Urdu text.

In [1]:
!pip install opencv-python-headless pillow

import cv2
import numpy as np
from PIL import Image
import os
import matplotlib.pyplot as plt

print("Libraries loaded successfully!")

Libraries loaded successfully!


In [2]:
def preprocess_image(image_path, save_path):
    # Load image
    img = cv2.imread(image_path)

    if img is None:
        print(f"Could not load: {image_path}")
        return

    # Step 1: Convert to grayscale
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    # Step 2: Resize image
    resized = cv2.resize(gray, (512, 128))

    # Step 3: Remove noise
    denoised = cv2.fastNlMeansDenoising(resized, h=10)

    # Step 4: Convert to black & white
    _, binary = cv2.threshold(denoised, 127, 255, cv2.THRESH_BINARY)

    # Save processed image
    cv2.imwrite(save_path, binary)

    return binary


# Create output folder
os.makedirs("data/processed", exist_ok=True)

print("Preprocessing function ready!")

Preprocessing function ready!


In [6]:
import glob

# Find all PNG images inside data/raw/
all_images = glob.glob("data/raw/**/*.png", recursive=True)

print(f"Found {len(all_images)} images to process")

processed_count = 0

for img_path in all_images:
    filename = os.path.basename(img_path)
    save_path = f"data/processed/{filename}"

    result = preprocess_image(img_path, save_path)

    if result is not None:
        processed_count += 1

print(f"Done! Processed {processed_count} images")
print("Check data/processed folder")

Found 100 images to process
Done! Processed 100 images
Check data/processed folder


In [5]:
!unzip -q data.zip

replace data/raw/books/64.png? [y]es, [n]o, [A]ll, [N]one, [r]ename: A


In [7]:
!apt-get install -y tesseract-ocr tesseract-ocr-urd

!pip install pytesseract

import pytesseract
from PIL import Image
import glob

print("Tesseract Installed Successfully!")

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
tesseract-ocr is already the newest version (4.1.1-2.1build1).
The following NEW packages will be installed:
  tesseract-ocr-urd
0 upgraded, 1 newly installed, 0 to remove and 3 not upgraded.
Need to get 1,000 kB of archives.
After this operation, 1,413 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/universe amd64 tesseract-ocr-urd all 1:4.00~git30-7274cfa-1.1 [1,000 kB]
Fetched 1,000 kB in 0s (12.5 MB/s)
Selecting previously unselected package tesseract-ocr-urd.
(Reading database ... 118243 files and directories currently installed.)
Preparing to unpack .../tesseract-ocr-urd_1%3a4.00~git30-7274cfa-1.1_all.deb ...
Unpacking tesseract-ocr-urd (1:4.00~git30-7274cfa-1.1) ...
Setting up tesseract-ocr-urd (1:4.00~git30-7274cfa-1.1) ...
Tesseract Installed Successfully!


In [8]:
# Test first 5 processed images

test_images = list(glob.glob("data/processed/*.png"))[:5]

print("=== Tesseract Results on Urdu Images ===\n")

for img_path in test_images:
    img = Image.open(img_path)

    # Urdu OCR
    result = pytesseract.image_to_string(img, lang="urd")

    print("Image:", img_path)
    print("Tesseract Output:")
    print(result)
    print("-" * 50)

=== Tesseract Results on Urdu Images ===

Image: data/processed/58.png
Tesseract Output:
ایور

لاہور: تجی یوٹیورسٹی کی طاليه کی 'خود کشی' کی
ت7ت ےس
کے مخابق ماقعہ بر خبدکامی کی

 
 
      

--------------------------------------------------
Image: data/processed/25.png
Tesseract Output:
ھی اگ ہو وا ول کیا اوران

--------------------------------------------------
Image: data/processed/2.png
Tesseract Output:
ا0 از

--------------------------------------------------
Image: data/processed/64.png
Tesseract Output:
 

--------------------------------------------------
Image: data/processed/65.png
Tesseract Output:
 

--------------------------------------------------


# Gap Analysis

### Image 58.png

**Actual Text:**
لاہور: نجی یونیورسٹی کی طالبہ کی 'خود کشی' کی کوشش، دوسری منزل سے چھلانگ لگا دی۔
پولیس کے مطابق واقعہ بظاہر خودکشی کی کوشش لگتا ہے، طالبہ کی نازک فریکچر ہو گئیں جبکہ یونیورسٹی نے آن کیمپس کلاسز معطل کر دیں۔

**Tesseract Output:**
ایور

لاہور: تجی یوٹیورسٹی کی طاليه کی 'خود کشی' کی
کے مخابق ماقعہ بر خبدکامی کی

**Issue:**
- "نجی" کو "تجی" پڑھا۔
- "طالبہ" کو "طاليه" پڑھا۔
- کئی الفاظ غلط recognize ہوئے۔
- پورا متن مکمل recognize نہیں ہوا۔
- آخری سطریں مکمل miss ہو گئیں.

---

### Image 25.png

**Actual Text:**
میں ہلاک ہونے والوں کی یاد میں اور ان کو

**Tesseract Output:**
ھی اگ ہو وا ول کیا اوران

**Issue:**
- زیادہ تر الفاظ غلط پہچانے گئے۔
- الفاظ کی ترتیب بھی غلط ہوگئی۔
- اصل جملہ صحیح طریقے سے recognize نہیں ہوا۔

---

### Image 2.png

**Actual Text:**
کی لوڈ شیڈنگ کے خلاف مظاہرہ کیا۔ پولیس تشدد سے ایک نوجوان

**Tesseract Output:**
ا0 از

**Issue:**
- تقریباً پورا متن recognize نہیں ہوا۔
- صرف چند غیر متعلقہ حروف detect ہوئے۔
- OCR اصل متن پڑھنے میں ناکام رہا۔

---

### Image 64.png

**Actual Text:**
عالیان ماں اپنے والدین کی اکلوتی اولاد تھی۔
اور اس کے والدین اس کی شادی سے پہلے وفات پا گئے۔
تو اس کی شادی کس نے کی اور ابلیشر کے ساتھ؟
ہمارے اور ان کے ماحول میں فرق ہے واحد۔

**Tesseract Output:**
(No Output)

**Issue:**
- OCR نے کوئی متن detect نہیں کیا۔
- مکمل image پڑھنے میں ناکام رہا۔

---

### Image 65.png

**Actual Text:**
گی، ماموں صدر سے 4 دن ہسپتال رہے۔ امرحہ علی کی چھت سے گر کر ٹانگ کی ہڈی ٹوٹ گئی۔۔۔۔

**Tesseract Output:**
(No Output)

**Issue:**
- OCR نے کوئی متن detect نہیں کیا۔
- مکمل image recognize نہیں ہوئی۔

---

# Summary

**Tesseract fails on Urdu because** it struggles to recognize connected Urdu characters, different fonts, low-quality images, and complex layouts. It often produces incorrect words, misses many characters, or completely fails to detect text. Therefore, a dedicated Urdu OCR model is required for accurate Urdu text recognition.